# Context RAG SandboxAgent

Chat directly through the SandboxAgent A2A endpoint. `chat()` opens a temporary port-forward and returns the unmodified JSON response.

Prerequisite: `kubectl` is installed and the current kube context can access the cluster.

In [13]:
import json
import subprocess
import time
import uuid
from urllib.request import Request, urlopen


def chat(agent: str, prompt: str, session: str | None = None):
    session = session or str(uuid.uuid4())
    message_id = str(uuid.uuid4())
    tunnel = subprocess.Popen(
        ["kubectl", "-n", "kagent", "port-forward",
         "service/kagent-controller", "18083:8083"],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
    )
    try:
        time.sleep(2)
        if tunnel.poll() is not None:
            raise RuntimeError(tunnel.stderr.read().strip())
        payload = {
            "jsonrpc": "2.0", "id": message_id, "method": "message/send",
            "params": {"message": {"kind": "message", "messageId": message_id,
                "role": "user", "contextId": session,
                "parts": [{"kind": "text", "text": prompt}]}},
        }
        request = Request(
            f"http://127.0.0.1:18083/api/a2a-sandboxes/kagent/{agent}/",
            data=json.dumps(payload).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urlopen(request, timeout=300) as response:
            return json.load(response)
    finally:
        tunnel.terminate()
        try:
            tunnel.wait(timeout=5)
        except subprocess.TimeoutExpired:
            tunnel.kill()

In [14]:
result = chat(
    "recsys-context-agent-sandbox",
    (
        'Call retrieve_rag_context exactly once with exactly '
        '{"query":"noise-cancelling headphones","top_k_items":1,"filters":null}. '
        'Then recommend the returned item in one sentence and cite its first chunk_id.'
    ),
)

result

{'jsonrpc': '2.0',
 'id': 'cdc551f9-b9eb-4839-bf7d-0b4af78d707f',
 'result': {'kind': 'task',
  'id': '01a0382a-c936-77c2-afe9-8a4b126c8312',
  'artifacts': [{'artifactId': '01a0382b-85a7-71a9-99a5-8734e0962c23',
    'parts': [{'kind': 'data',
      'data': {'id': 'dFa5pTGmepXgkTDNSLaI3oCOq4hwNCMF',
       'name': 'retrieve_rag_context',
       'response': {'output': {'items': [{'average_rating': 4.9,
           'brand': 'Sennheiser',
           'category_path': ['Điện tử', 'Gaming', 'Tai nghe gaming'],
           'current_price': 29.99,
           'evidence': [{'chunk_id': '800059:product_overview:overview:0',
             'chunk_type': 'product_overview',
             'score': 0.8261429071426392,
             'source_key': 'overview',
             'text': 'Sennheiser Continuous 800059 - Tai nghe gaming\nTai nghe gaming Sennheiser Continuous 800059 được thiết kế dành cho game thủ với chất âm vòm ảo sống động, micro chống ồn rõ ràng và đệm tai êm ái. Phù hợp cho các buổi chơi game dài 

### Conclusion from the recorded result

The result above confirms that `recsys-context-agent-sandbox` invoked the `retrieve_rag_context` MCP tool and received RAG data from the function:

- The first `adk_type: function_call` event omitted its arguments, so MCP returned a `function_response` containing the validation error `query: Field required`.
- The agent then corrected the request and issued a `function_call` with `query="noise-cancelling headphones"`, `top_k_items=1`, and `filters=None`.
- The matching `function_response`, with tool-call ID `dFa5pTGmepXgkTDNSLaI3oCOq4hwNCMF`, successfully returns item `800059`, `pipeline_run_id=rag-pipeline-fallback96-20260823`, and evidence chunk `800059:product_overview:overview:0`.
- `status.state: completed` confirms that the A2A task finished. The successful `function_call`/`function_response` pair directly demonstrates the `SandboxAgent -> RemoteMCPServer -> retrieve_rag_context` execution path.

Experiment caveat: the prompt requested exactly one call, but the trace shows two attempts—one failed call without arguments followed by one successful retry. This still demonstrates working MCP execution while documenting tool-call generation behavior that should be improved. The notebook preserves the raw A2A response without a parser or intermediary helper.